# Calibrated ticket triage in 10 lines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PunkyTigerLabs/superion-s1/blob/main/notebooks/calibrated_ticket_triage.ipynb)

**Superion S1** by Punky Tiger Labs answers typed questions about a text — yes/no, choice, score — with
**calibrated probabilities**, in one API call, without generating text (~0.3 s p50).

**#1 of 213 on S1MB English v1 — Task Average 75.06 · Borda 89.88 (Nov 2026)** ·
[leaderboard](https://huggingface.co/datasets/hotchpotch/s1mb-result)

This notebook routes support tickets to a team, rates urgency and flags refund requests — and, because the
probabilities are calibrated, sends only the uncertain ones to a human.

**You need a free key** (100 M input tokens per day, permanent): https://punkytigerlabs.com/s1/
In Colab, store it as a secret named `SUPERION_API_KEY` (🔑 icon on the left) and allow notebook access.

In [ ]:
%pip install -q superion-s1 pandas

In [ ]:
import os
if not os.environ.get("SUPERION_API_KEY"):
    try:
        from google.colab import userdata          # Colab secret
        os.environ["SUPERION_API_KEY"] = userdata.get("SUPERION_API_KEY")
    except Exception:
        import getpass
        os.environ["SUPERION_API_KEY"] = getpass.getpass("Superion S1 key (s1k_…): ")

## The 10 lines

In [ ]:
from superion_s1 import Superion, choice, noul, score
import pandas as pd

tickets = [
    "My card was charged twice for order #4411. Please fix it today.",
    "How do I export my invoices to CSV?",
    "The app crashes every time I open the camera on Android 15.",
    "Cancel my subscription and refund the last month, I never used it.",
    "Your new pricing page is confusing, which plan includes SSO?",
    "Hi, just wanted to say the new dashboard is great!",
]
s1 = Superion()
rows = [s1.decide(f"Ticket: {t}", {
    "team": choice("Which team should handle this ticket?",
                   {"billing": "billing, payments, refunds", "tech": "technical bug or crash",
                    "sales": "plans, pricing, upgrades", "howto": "how-to question", "other": "anything else"}),
    "urgency": score("How urgent is a reply?", ["low", "normal", "high"]),
    "refund": noul("Does the customer ask for money back?"),
}) for t in tickets]

df = pd.DataFrame({"ticket": tickets,
                   "team": [r["team"].best for r in rows], "p_team": [r["team"].p for r in rows],
                   "urgency": [r["urgency"].best_label for r in rows], "p_refund": [r["refund"].p for r in rows]})
df

## Route with a threshold that means what it says

Calibrated means a 0.9 is right about 9 times out of 10. So pick the confidence you need and let
everything below it go to a person.

In [ ]:
THRESHOLD = 0.8
df["route"] = [team if p >= THRESHOLD else "human-review" for team, p in zip(df.team, df.p_team)]
df[["ticket", "route", "p_team", "urgency", "p_refund"]]

## What did that cost?

In [ ]:
tokens = sum(r.usage.input_tokens for r in rows)
print(f"{len(rows)} tickets, 3 decisions each, {tokens} input tokens")
print(f"Prepaid price: {tokens * 0.042 / 1e6:.6f} USD  (Free plan: 100 M tokens/day at no cost)")
print(f"≈ {1e6 / (0.042 * tokens / len(rows)):,.0f} tickets per USD at this size")

## Next

* Docs: https://punkytigerlabs.com/s1/docs · Python SDK: `pip install superion-s1` · JS: `npm i @punkytigerlabs/superion-s1`
* Integrations: LangChain (`langchain-superion`), LlamaIndex (`llama-index-tools-superion`), n8n (`n8n-nodes-superion-s1`)

**Custom & industry models.** Punky Tiger Labs also builds custom and industry-specific decision models —
fitted to your own taxonomy, labels and data, for a specific industry or objective — served on a dedicated
instance. Contact: alvaro.lopez@punkytigerlabs.com